In [ ]:
# P11: Convolutional Autoencoder on MNIST Dataset
# Code by Parthiv Abhani

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow Version:", tf.__version__)


# ==========================================
# 1. Load MNIST Dataset
# ==========================================
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)


# ==========================================
# 2. Preprocess Images
# ==========================================
# Normalize pixel values to range 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("After preprocessing:", x_train.shape)


# ==========================================
# 3. Build Convolutional Autoencoder
# ==========================================

# Encoder
encoder = models.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2), padding="same"),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2), padding="same")
], name="Encoder")


# Decoder
decoder = models.Sequential([
    layers.Input(shape=(7, 7, 16)),

    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.UpSampling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.UpSampling2D((2, 2)),

    layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same")
], name="Decoder")


# Complete Autoencoder
autoencoder = models.Sequential([
    encoder,
    decoder
], name="Convolutional_Autoencoder")


# ==========================================
# 4. Compile Model
# ==========================================
autoencoder.compile(
    optimizer="adam",
    loss="binary_crossentropy"
)

autoencoder.summary()


# ==========================================
# 5. Train Autoencoder
# ==========================================
history = autoencoder.fit(
    x_train,
    x_train,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    shuffle=True
)


# ==========================================
# 6. Evaluate Model
# ==========================================
test_loss = autoencoder.evaluate(
    x_test,
    x_test,
    verbose=0
)

print("\nTest Reconstruction Loss:", round(test_loss, 4))


# ==========================================
# 7. Reconstruct Test Images
# ==========================================
reconstructed_images = autoencoder.predict(x_test)


# ==========================================
# 8. Compare Original and Reconstructed Images
# ==========================================
n = 10

plt.figure(figsize=(15, 4))

for i in range(n):

    # Original image
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title("Original")
    plt.axis("off")

    # Reconstructed image
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(reconstructed_images[i].squeeze(), cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")

plt.tight_layout()
plt.show()


# ==========================================
# 9. Plot Training Loss
# ==========================================
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Autoencoder Training Loss")
plt.legend()
plt.show()


# ==========================================
# 10. Display Encoder Output
# ==========================================
encoded_images = encoder.predict(x_test[:10])

print("Original image shape:", x_test[0].shape)
print("Compressed representation shape:", encoded_images[0].shape)